# Tweet Sentiment Analysis with LSTM Models

This notebook compares three LSTM architectures for sentiment classification:
- LSTM
- Bidirectional LSTM
- Encoder-Decoder LSTM

**Dataset**: CardiffNLP Tweet Sentiment Evaluation (3-class: negative, neutral, positive)

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.tensorflow

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    TextVectorization, Embedding, LSTM, Bidirectional,
    Dense, Dropout, RepeatVector, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping

## 2. Configuration and Reproducibility

In [2]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

# Configuration
CONFIG = {
    "max_words": 10000,
    "seq_len": 50,
    "embedding_dim": 64,
    "lstm_units": 64,
    "dropout_rate": 0.3,
    "epochs": 10,
    "batch_size": 64,
    "validation_split": 0.1,
    "test_size": 0.2,
    "optimizer": "adam",
    "loss_function": "sparse_categorical_crossentropy",
    "random_state": 42,
}

# MLflow experiment setup
mlflow.set_experiment("Tweet_Sentiment_LSTM_Models")

<Experiment: artifact_location='/Users/juliakrempinska/Documents/PWR/masters/1_sem/machine_learning/project/notebooks/mlruns/3', creation_time=1780499586239, experiment_id='3', last_update_time=1780499586239, lifecycle_stage='active', name='Tweet_Sentiment_LSTM_Models', tags={}, workspace='default'>

## 3. Load and Explore Data

In [4]:
# Load dataset
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
df = pd.DataFrame(dataset["train"])

# Basic exploration
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few examples:")
display(df.head())

Dataset shape: (45615, 2)

First few examples:


,text,label
0,"""QT @user In the original draft of the 7th boo...",2
1,"""Ben Smith / Smith (concussion) remains out of...",1
2,Sorry bout the stream last night I crashed out...,1
3,Chase Headley's RBI double in the 8th inning o...,1
4,@user Alciato: Bee will invest 150 million in ...,2


## 4. Data Preprocessing

In [5]:
# Extract features and labels
X = df["text"].astype(str).values
y = df["label"].astype(int).values

# Train/test split with stratification (best practice for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Training set label distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

Training set size: 36492
Test set size: 9123
Training set label distribution:
0     5674
1    16539
2    14279
Name: count, dtype: int64


## 5. Text Vectorization

In [6]:
# Create and fit text vectorizer
vectorizer = TextVectorization(
    max_tokens=CONFIG["max_words"],
    output_sequence_length=CONFIG["seq_len"]
)

# Adapt vectorizer only on training data (best practice to prevent data leakage)
vectorizer.adapt(X_train)

## 6. Define Training Function

In [7]:
def train_and_evaluate(model, name):
    """
    Train and evaluate a model, logging results to MLflow.
    
    Args:
        model: Compiled TensorFlow/Keras model
        name: String identifier for the model run
        
    Returns:
        accuracy: Test accuracy score
    """
    with mlflow.start_run(run_name=name):
        # Log parameters to MLflow
        mlflow.log_params({
            "model_name": name,
            "max_words": CONFIG["max_words"],
            "seq_len": CONFIG["seq_len"],
            "embedding_dim": CONFIG["embedding_dim"],
            "lstm_units": CONFIG["lstm_units"],
            "dropout_rate": CONFIG["dropout_rate"],
            "epochs": CONFIG["epochs"],
            "batch_size": CONFIG["batch_size"],
            "optimizer": CONFIG["optimizer"],
            "loss_function": CONFIG["loss_function"],
        })

        # Early stopping callback
        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )

        # Compile model
        model.compile(
            optimizer=CONFIG["optimizer"],
            loss=CONFIG["loss_function"],
            metrics=["accuracy"]
        )

        # Train the model
        history = model.fit(
            X_train,
            y_train,
            epochs=CONFIG["epochs"],
            batch_size=CONFIG["batch_size"],
            validation_split=CONFIG["validation_split"],
            callbacks=[early_stop],
            verbose=1
        )

        # Evaluate on test set
        test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
        y_pred = np.argmax(model.predict(X_test), axis=1)
        acc = accuracy_score(y_test, y_pred)

        # Log metrics to MLflow
        mlflow.log_metrics({
            "test_loss": test_loss,
            "test_accuracy": test_accuracy,
            "accuracy_score": acc,
        })

        # Log training history metrics
        for epoch, value in enumerate(history.history["loss"]):
            mlflow.log_metric("train_loss", value, step=epoch)
        for epoch, value in enumerate(history.history["accuracy"]):
            mlflow.log_metric("train_accuracy", value, step=epoch)
        for epoch, value in enumerate(history.history["val_loss"]):
            mlflow.log_metric("val_loss", value, step=epoch)
        for epoch, value in enumerate(history.history["val_accuracy"]):
            mlflow.log_metric("val_accuracy", value, step=epoch)

        # Log the model artifact
        mlflow.tensorflow.log_model(
            model=model,
            artifact_path=f"{name}_model"
        )

        print(f"\n{'='*50}")
        print(f"Model: {name}")
        print(f"Test Accuracy: {acc:.4f}")
        print(f"Test Loss: {test_loss:.4f}")
        print(f"{'='*50}")

        return y_pred, acc

## 7. Define Model Architectures

In [8]:
# Standard LSTM Model
lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=CONFIG["max_words"], output_dim=CONFIG["embedding_dim"]),
    LSTM(CONFIG["lstm_units"]),
    Dropout(CONFIG["dropout_rate"]),
    Dense(3, activation="softmax")
])

print("LSTM Model Architecture:")
lstm_model.summary()

LSTM Model Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Bidirectional LSTM Model
bilstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=CONFIG["max_words"], output_dim=CONFIG["embedding_dim"]),
    Bidirectional(LSTM(CONFIG["lstm_units"])),
    Dropout(CONFIG["dropout_rate"]),
    Dense(3, activation="softmax")
])

print("\nBidirectional LSTM Model Architecture:")
bilstm_model.summary()


Bidirectional LSTM Model Architecture:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Encoder-Decoder LSTM Model
ed_lstm_model = Sequential([
    vectorizer,
    Embedding(input_dim=CONFIG["max_words"], output_dim=CONFIG["embedding_dim"]),
    
    # Encoder
    LSTM(CONFIG["lstm_units"]),
    
    # Decoder
    RepeatVector(CONFIG["seq_len"]),
    LSTM(CONFIG["lstm_units"], return_sequences=True),
    
    # Classification
    GlobalAveragePooling1D(),
    Dropout(CONFIG["dropout_rate"]),
    Dense(3, activation="softmax")
])

print("\nEncoder-Decoder LSTM Model Architecture:")
ed_lstm_model.summary()


Encoder-Decoder LSTM Model Architecture:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 8. Train and Evaluate Models

In [11]:
# Train models and store results
results = {}
all_predictions = {}

lstm_pred, lstm_acc = train_and_evaluate(lstm_model, "LSTM")
results["LSTM"] = lstm_acc
all_predictions["LSTM"] = lstm_pred

bilstm_pred, bilstm_acc = train_and_evaluate(bilstm_model, "Bi-LSTM")
results["Bi-LSTM"] = bilstm_acc
all_predictions["Bi-LSTM"] = bilstm_pred

ed_lstm_pred, ed_lstm_acc = train_and_evaluate(ed_lstm_model, "ED-LSTM")
results["ED-LSTM"] = ed_lstm_acc
all_predictions["ED-LSTM"] = ed_lstm_pred

Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4481 - loss: 1.0195 - val_accuracy: 0.4507 - val_loss: 1.0173
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.4481 - loss: 1.0195 - val_accuracy: 0.4507 - val_loss: 1.0173
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.4807 - loss: 0.9514 - val_accuracy: 0.5877 - val_loss: 0.8539
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.4807 - loss: 0.9514 - val_accuracy: 0.5877 - val_loss: 0.8539
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6434 - loss: 0.7820 - val_accuracy: 0.5789 - val_loss: 0.8614
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6434 - loss: 0.7820 - val_accuracy: 0.5789 - val_loss: 0.8614
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7165 - loss: 0.6788 - val_accuracy: 0.6164 - val_loss: 0.9217
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7165 - loss: 0.6788 - val_accuracy: 0.6164 - val_

2026/06/03 17:27:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 17:27:49 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/06/03 17:27:49 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Model: LSTM
Test Accuracy: 0.5804
Test Loss: 0.8613
Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5903 - loss: 0.8576 - val_accuracy: 0.6342 - val_loss: 0.7846
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5903 - loss: 0.8576 - val_accuracy: 0.6342 - val_loss: 0.7846
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7092 - loss: 0.6616 - val_accuracy: 0.6279 - val_loss: 0.8028
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7092 - loss: 0.6616 - val_accuracy: 0.6279 - val_loss: 0.8028
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7555 - loss: 0.5716 - val_accuracy: 0.6205 - val_loss: 0.9086
514/514 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7555 - loss: 0.5716 - val_accuracy: 0.6205 - val_loss: 0.9086
286/286 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
286/286 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


2026/06/03 17:28:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 17:28:09 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/06/03 17:28:09 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Model: Bi-LSTM
Test Accuracy: 0.6370
Test Loss: 0.7839
Epoch 1/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.4461 - loss: 1.0194 - val_accuracy: 0.4507 - val_loss: 1.0169
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.4461 - loss: 1.0194 - val_accuracy: 0.4507 - val_loss: 1.0169
Epoch 2/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.4503 - loss: 1.0093 - val_accuracy: 0.4619 - val_loss: 0.9660
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.4503 - loss: 1.0093 - val_accuracy: 0.4619 - val_loss: 0.9660
Epoch 3/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.4698 - loss: 0.9277 - val_accuracy: 0.5211 - val_loss: 0.8985
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.4698 - loss: 0.9277 - val_accuracy: 0.5211 - val_loss: 0.8985
Epoch 4/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.6001 - loss: 0.8226 - val_accuracy: 0.5710 - val_loss: 0.8567
Epoch 5/10
514/514 ━━━━━━━━━━━━━━━━━━━━ 8

2026/06/03 17:29:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 17:29:00 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/06/03 17:29:00 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.



Model: ED-LSTM
Test Accuracy: 0.5785
Test Loss: 0.8704


## 9. Compare Results

In [12]:
# Display overall results
results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])

print("\n" + "="*50)
print("Overall Model Comparison")
print("="*50)
print(results_df.to_string(index=False))


Overall Model Comparison
  Model  Accuracy
   LSTM  0.580401
Bi-LSTM  0.636962
ED-LSTM  0.578538


## 10. Detailed Model Evaluation

In [13]:
# Detailed evaluation of all models
label_names = ["negative", "neutral", "positive"]

for model_name, y_pred in all_predictions.items():
    print("\n" + "="*50)
    print(model_name)
    print("="*50)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        zero_division=0
    ))

    # Confusion matrix analysis
    cm = confusion_matrix(y_test, y_pred)
    summary = pd.DataFrame({
        "Class": label_names,
        "Correctly Predicted": cm.diagonal(),
        "Total in Test Set": cm.sum(axis=1),
        "Accuracy per Class": cm.diagonal() / cm.sum(axis=1)
    })

    print("\nDetailed Class Summary:")
    print(summary)


LSTM

Classification Report:
              precision    recall  f1-score   support

    negative       0.50      0.32      0.39      1419
     neutral       0.54      0.76      0.63      4134
    positive       0.71      0.47      0.57      3570

    accuracy                           0.58      9123
   macro avg       0.58      0.52      0.53      9123
weighted avg       0.60      0.58      0.57      9123


Detailed Class Summary:
      Class  Correctly Predicted  Total in Test Set  Accuracy per Class
0  negative                  452               1419            0.318534
1   neutral                 3160               4134            0.764393
2  positive                 1683               3570            0.471429

Bi-LSTM

Classification Report:
              precision    recall  f1-score   support

    negative       0.51      0.33      0.40      1419
     neutral       0.63      0.67      0.65      4134
    positive       0.67      0.72      0.70      3570

    accuracy             